In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

RAW_SEC_DIR = DATA_DIR / "raw" / "sec_filings"
RAW_EARNINGS_DIR = DATA_DIR / "raw" / "earnings_calls"
PROCESSED_DIR = DATA_DIR / "processed"
LABELED_DIR = DATA_DIR / "labeled"

RAW_SEC_DIR.mkdir(parents=True, exist_ok=True)
RAW_EARNINGS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
LABELED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import re
import json
import time
import requests
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup

BASE = Path("/Users/yiqian/src/hedge-fund-research-assistant")
RAW_DIR = BASE / "data" / "raw"
PROCESSED_DIR = BASE / "data" / "processed"

HEADERS = {
    "User-Agent": "yiqian academic project yiqian1999@outlook.com"
}

TICKERS = ["MU", "MRVL", "NBIS"] # get tickers for the three stocks

In [5]:
#map tickers to CIKs using the SEC's company_tickers.json file
def load_ticker_cik_map():
    url = "https://www.sec.gov/files/company_tickers.json"
    data = requests.get(url, headers=HEADERS).json()

    rows = []
    for _, item in data.items():
        rows.append({
            "ticker": item["ticker"].upper(),
            "title": item["title"],
            "cik": str(item["cik_str"]).zfill(10),
        })

    return pd.DataFrame(rows)

ticker_map = load_ticker_cik_map()
ticker_map[ticker_map["ticker"].isin(TICKERS)]

,ticker,title,cik
8,MU,MICRON TECHNOLOGY INC,0000723125
55,MRVL,"Marvell Technology, Inc.",0001835632
243,NBIS,Nebius Group N.V.,0001513845


In [7]:
# get recent filing
def get_submissions(cik):
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    r = requests.get(url, headers=HEADERS)
    r.raise_for_status()
    return r.json()

def recent_filings_for_ticker(ticker):
    cik = ticker_map.loc[ticker_map["ticker"] == ticker, "cik"].iloc[0]
    sub = get_submissions(cik)
    recent = pd.DataFrame(sub["filings"]["recent"])
    recent["ticker"] = ticker
    recent["cik"] = cik
    return recent

filings = pd.concat([recent_filings_for_ticker(t) for t in TICKERS])
filings[["ticker", "form", "filingDate", "accessionNumber", "primaryDocument"]].head()
# filter 
target_forms = {
    "MU": ["10-K", "10-Q", "8-K"],
    "MRVL": ["10-K", "10-Q", "8-K"],
    "NBIS": ["20-F", "6-K"],
}

target_filings = pd.concat([
    filings[(filings["ticker"] == ticker) & (filings["form"].isin(forms))]
    for ticker, forms in target_forms.items()
])

target_filings[["ticker", "form", "filingDate", "accessionNumber", "primaryDocument"]].head(20)

,ticker,form,filingDate,accessionNumber,primaryDocument
0,MU,10-Q,2026-06-25,0000723125-26-000015,mu-20260528.htm
1,MU,8-K,2026-06-24,0000723125-26-000013,mu-20260624.htm
4,MU,8-K,2026-06-09,0001104659-26-071845,tm2617112d1_8k.htm
30,MU,8-K,2026-04-01,0001104659-26-038249,tm2610810d1_8k.htm
32,MU,8-K,2026-03-25,0001104659-26-034174,tm269755d1_8k.htm
33,MU,10-Q,2026-03-19,0000723125-26-000006,mu-20260226.htm
34,MU,8-K,2026-03-18,0000723125-26-000004,mu-20260318.htm
43,MU,8-K,2026-01-21,0001104659-26-005366,tm263707d1_8k.htm
56,MU,10-Q,2025-12-18,0000723125-25-000046,mu-20251127.htm
57,MU,8-K,2025-12-17,0000723125-25-000044,mu-20251217.htm


In [8]:
# Download filing 
def filing_url(row):
    cik_int = str(int(row["cik"]))
    accession = row["accessionNumber"].replace("-", "")
    doc = row["primaryDocument"]
    return f"https://www.sec.gov/Archives/edgar/data/{cik_int}/{accession}/{doc}"

def download_filing(row):
    ticker = row["ticker"]
    form = row["form"]
    date = row["filingDate"]
    accession = row["accessionNumber"]

    out_dir = RAW_DIR / "sec" / ticker
    out_dir.mkdir(parents=True, exist_ok=True)

    path = out_dir / f"{date}_{form}_{accession}.html"
    if path.exists():
        return path

    url = filing_url(row)
    r = requests.get(url, headers=HEADERS)
    r.raise_for_status()

    path.write_text(r.text, encoding="utf-8")
    time.sleep(0.2)
    return path

sample = target_filings.groupby(["ticker", "form"]).head(3).copy()
sample["local_path"] = sample.apply(download_filing, axis=1)
sample[["ticker", "form", "filingDate", "local_path"]]

,ticker,form,filingDate,local_path
0,MU,10-Q,2026-06-25,/Users/yiqian/src/hedge-fund-research-assistan...
1,MU,8-K,2026-06-24,/Users/yiqian/src/hedge-fund-research-assistan...
4,MU,8-K,2026-06-09,/Users/yiqian/src/hedge-fund-research-assistan...
30,MU,8-K,2026-04-01,/Users/yiqian/src/hedge-fund-research-assistan...
33,MU,10-Q,2026-03-19,/Users/yiqian/src/hedge-fund-research-assistan...
56,MU,10-Q,2025-12-18,/Users/yiqian/src/hedge-fund-research-assistan...
103,MU,10-K,2025-10-03,/Users/yiqian/src/hedge-fund-research-assistan...
250,MU,10-K,2024-10-04,/Users/yiqian/src/hedge-fund-research-assistan...
409,MU,10-K,2023-10-06,/Users/yiqian/src/hedge-fund-research-assistan...
0,MRVL,8-K,2026-06-25,/Users/yiqian/src/hedge-fund-research-assistan...


In [10]:
#extract text from filing
def html_to_text(path):
    html = Path(path).read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(html, "lxml")

    for tag in soup(["script", "style", "table"]):
        tag.decompose()

    text = soup.get_text("\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

sample["text"] = sample["local_path"].apply(html_to_text)
sample[["ticker", "form", "filingDate", "text"]].head()

# extract useful sections from 10-K and 10-Q filings
def extract_between(text, start_patterns, end_patterns):
    lower = text.lower()

    starts = []
    for pat in start_patterns:
        m = re.search(pat, lower, flags=re.I)
        if m:
            starts.append(m.start())

    if not starts:
        return None

    start = min(starts)
    tail = lower[start:]

    ends = []
    for pat in end_patterns:
        m = re.search(pat, tail, flags=re.I)
        if m and m.start() > 500:
            ends.append(start + m.start())

    end = min(ends) if ends else min(len(text), start + 120_000)
    return text[start:end].strip()

SECTION_RULES = {
    "risk_factors": {
        "start": [r"item\s+1a\.?\s+risk\s+factors", r"item\s+3d\.?\s+risk\s+factors"],
        "end": [r"item\s+1b\.?", r"item\s+2\.?", r"item\s+4\.?"],
    },
    "mda": {
        "start": [r"item\s+7\.?\s+management", r"item\s+2\.?\s+management", r"item\s+5\.?\s+operating"],
        "end": [r"item\s+7a\.?", r"item\s+3\.?", r"item\s+6\.?"],
    },
    "market_risk": {
        "start": [r"item\s+7a\.?\s+quantitative", r"item\s+3\.?\s+quantitative"],
        "end": [r"item\s+8\.?", r"item\s+4\.?"],
    },
}

def extract_sections(text):
    sections = {}
    for name, rules in SECTION_RULES.items():
        sections[name] = extract_between(text, rules["start"], rules["end"])
    return sections

/var/folders/sm/tsjy_qyx34ldwx8ksz604qv00000gn/T/ipykernel_59047/1823717485.py:4: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")


In [11]:
#chunk text
def chunk_words(text, chunk_size=220, overlap=40):
    if not text:
        return []

    words = text.split()
    chunks = []

    step = chunk_size - overlap
    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size])
        if len(chunk.split()) >= 80:
            chunks.append(chunk)

    return chunks

In [12]:
#build first dataset
records = []

for _, row in sample.iterrows():
    sections = extract_sections(row["text"])

    for section_name, section_text in sections.items():
        for i, chunk in enumerate(chunk_words(section_text)):
            records.append({
                "chunk_id": f"{row['ticker']}_{row['form']}_{row['filingDate']}_{section_name}_{i:04d}",
                "ticker": row["ticker"],
                "form": row["form"],
                "filing_date": row["filingDate"],
                "section": section_name,
                "source_path": str(row["local_path"]),
                "text": chunk,
            })

chunks = pd.DataFrame(records)
chunks.to_csv(PROCESSED_DIR / "sec_chunks_initial.csv", index=False)
chunks.head()

,chunk_id,ticker,form,filing_date,section,source_path,text
0,MU_10-Q_2026-06-25_risk_factors_0000,MU,10-Q,2026-06-25,risk_factors,/Users/yiqian/src/hedge-fund-research-assistan...,Item 1A. Risk Factors. 4 Table of Contents PAR...
1,MU_10-Q_2026-06-25_risk_factors_0001,MU,10-Q,2026-06-25,risk_factors,/Users/yiqian/src/hedge-fund-research-assistan...,"in millions, except per share amounts) (Unaudi..."
2,MU_10-Q_2026-06-25_risk_factors_0002,MU,10-Q,2026-06-25,risk_factors,/Users/yiqian/src/hedge-fund-research-assistan...,current-period presentation. Our fiscal year i...
3,MU_10-Q_2026-06-25_risk_factors_0003,MU,10-Q,2026-06-25,risk_factors,/Users/yiqian/src/hedge-fund-research-assistan...,of certain expenses in the notes to the financ...
4,MU_10-Q_2026-06-25_risk_factors_0004,MU,10-Q,2026-06-25,risk_factors,/Users/yiqian/src/hedge-fund-research-assistan...,"either on a modified prospective, modified ret..."


In [ ]:
LABELS = [
    "Demand / Revenue Growth",
    "Margins / Profitability",
    "Regulation / Legal",
    "Product / Technology Strategy",
    "Macro / Cyclical Risk",
    "Macro Risk",
    "Neutral / Other",
]

WEAK_LABEL_KEYWORDS = {
    "Demand Growth": [
        "demand",
        "customer demand",
        "strong demand",
        "end-market demand",
        "growth",
        "revenue growth",
        "sales growth",
        "bookings",
        "backlog",
        "order growth",
        "customer adoption",
        "adoption",
        "market opportunity",
        "market expansion",
        "secular growth",
        "tailwinds",
        "new customers",
        "design wins",
        "unit growth",
    ],

    "Margins / Profitability": [
        "margin",
        "gross margin",
        "operating margin",
        "margin pressure",
        "pricing pressure",
        "price erosion",
        "cost pressure",
        "higher costs",
        "input costs",
        "supply chain costs",
        "product mix",
        "profitability",
        "net income",
        "operating income",
        "gross profit",
        "operating profit",
        "earnings",
        "eps",
        "ebitda",
        "operating leverage",
        "cost reduction",
        "cost savings",
        "free cash flow",
    ],

    "Competition": [
        "competition",
        "competitive",
        "competitor",
        "competitors",
        "compete",
        "market share",
        "share loss",
        "pricing competition",
        "competitive pressure",
        "substitute products",
        "alternative solutions",
        "new entrants",
        "custom silicon",
        "in-house chips",
        "customer-developed chips",
        "rival",
    ],

    "Regulation / Legal": [
        "regulation",
        "regulatory",
        "legal",
        "lawsuit",
        "litigation",
        "claim",
        "investigation",
        "compliance",
        "export control",
        "export controls",
        "trade restrictions",
        "sanctions",
        "government restrictions",
        "antitrust",
        "intellectual property",
        "patent",
        "data privacy",
        "tax law",
        "tariff",
    ],

    "Capital Allocation / CAPEX": [
        "capital allocation",
        "capital expenditures",
        "capex",
        "capital investment",
        "investment in capacity",
        "capacity expansion",
        "data center investment",
        "fab investment",
        "manufacturing capacity",
        "equipment purchases",
        "infrastructure investment",
        "facility expansion",
        "construction",
        "research and development",
        "r&d",
        "technology investment",
        "share repurchase",
        "repurchase program",
        "buyback",
        "dividend",
        "debt repayment",
        "acquisition",
        "merger",
        "m&a",
        "liquidity",
        "balance sheet",
    ],

    "AI / Product Strategy": [
        "artificial intelligence",
        " ai ",
        "generative ai",
        "machine learning",
        "accelerated computing",
        "gpu",
        "accelerator",
        "data center",
        "cloud",
        "product roadmap",
        "roadmap",
        "platform",
        "architecture",
        "product launch",
        "new product",
        "innovation",
        "inference",
        "training workloads",
        "neural network",
        "large language model",
        "llm",
        "ai infrastructure",
    ],

    "Macro Risk": [
        "macroeconomic",
        "macro",
        "recession",
        "slowdown",
        "inflation",
        "interest rates",
        "foreign exchange",
        "fx",
        "currency",
        "geopolitical",
        "china",
        "tariff",
        "supply constraints",
        "supply shortage",
        "economic uncertainty",
        "consumer spending",
        "enterprise spending",
        "semiconductor cycle",
        "cyclical",
    ],

    "Neutral / Other": [],
}

In [15]:
# weak labeling
def weak_label(text):
    text_l = f" {text.lower()} "
    scores = {}

    for label, keywords in WEAK_LABEL_KEYWORDS.items():
        scores[label] = sum(keyword.lower() in text_l for keyword in keywords)

    best_label = max(scores, key=scores.get)
    best_score = scores[best_label]

    if best_score == 0:
        return "Neutral / Other", 0

    return best_label, best_score

chunks[["weak_label", "weak_label_score"]] = chunks["text"].apply(
    lambda x: pd.Series(weak_label(x))
)

chunks.to_csv(BASE / "data" / "labeled" / "weak_labeled_chunks_initial.csv", index=False)
chunks["weak_label"].value_counts()

weak_label
Regulation / Legal            618
Demand Growth                 298
Capital Allocation / CAPEX    282
Competition                   227
Margins / Profitability       215
Neutral / Other               135
Macro Risk                    127
AI / Product Strategy         108
Name: count, dtype: int64

The following code blocks are used to create manually reviewed labeled data

In [17]:
review_df = chunks.copy()

review_df["manual_label"] = review_df["weak_label"]
review_df["review_status"] = "unreviewed"
review_df["review_notes"] = ""

review_df.to_csv(
    BASE / "data" / "labeled" / "manual_review_initial.csv",
    index=False
)
review_sample = (
    chunks
    .groupby("weak_label", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), 50), random_state=42))
)

review_sample["manual_label"] = review_sample["weak_label"]
review_sample["review_status"] = "unreviewed"
review_sample["review_notes"] = ""

review_sample.to_csv(
    BASE / "data" / "labeled" / "manual_review_sample_500.csv",
    index=False
)

/var/folders/sm/tsjy_qyx34ldwx8ksz604qv00000gn/T/ipykernel_59047/130296270.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), 50), random_state=42))


Split data

In [19]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

BASE = Path("/Users/yiqian/src/hedge-fund-research-assistant")

df = pd.read_csv(BASE / "data/labeled/manual_review_taxonomy_final.csv")

df = df[df["review_status"] == "reviewed"].copy()
df = df[df["manual_label"].notna()].copy()
df = df[df["text"].notna()].copy()

model_df = df[
    [
        "chunk_id",
        "ticker",
        "form",
        "filing_date",
        "section",
        "text",
        "manual_label",
        "secondary_label",
        "source_path",
    ]
].copy()

model_df = model_df.rename(columns={"manual_label": "label"})

train_df, temp_df = train_test_split(
    model_df,
    test_size=0.30,
    stratify=model_df["label"],
    random_state=42,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42,
)

out_dir = BASE / "data/splits"
out_dir.mkdir(parents=True, exist_ok=True)

train_df.to_csv(out_dir / "train.csv", index=False)
val_df.to_csv(out_dir / "validation.csv", index=False)
test_df.to_csv(out_dir / "test.csv", index=False)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

LABEL_COL = "label"
print("\nTrain label distribution:")
print(train_df[LABEL_COL].value_counts(normalize=True).round(3))

print("\nValidation label distribution:")
print(val_df[LABEL_COL].value_counts(normalize=True).round(3))

print("\nTest label distribution:")
print(test_df[LABEL_COL].value_counts(normalize=True).round(3))

Train: (1407, 9)
Validation: (301, 9)
Test: (302, 9)

Train label distribution:
label
Regulation / Legal            0.259
Capital Allocation / CAPEX    0.210
Macro Risk                    0.123
AI / Product Strategy         0.117
Neutral / Other               0.092
Margins / Profitability       0.090
Competition                   0.060
Demand Growth                 0.048
Name: proportion, dtype: float64

Validation label distribution:
label
Regulation / Legal            0.259
Capital Allocation / CAPEX    0.209
Macro Risk                    0.123
AI / Product Strategy         0.116
Margins / Profitability       0.090
Neutral / Other               0.090
Competition                   0.063
Demand Growth                 0.050
Name: proportion, dtype: float64

Test label distribution:
label
Regulation / Legal            0.258
Capital Allocation / CAPEX    0.212
Macro Risk                    0.123
AI / Product Strategy         0.119
Neutral / Other               0.093
Margins / Profitabilit